In [1]:
import os
import pandas as pd
from prettytable import PrettyTable

from lib.uncertainty import Simulate

/Users/felix/MSE/03_projects/MT/zz_code/04_experiments/03_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DS_FEVER = '../02_data/2026-03-12/fever'
DS_HOTPOTQA = '../02_data/2026-03-12/hotpotqa'
DS_NQ = '../02_data/2026-03-12/nq'

In [6]:
def get_retrieval_success(row):
    ret_set = set([f"{r['document_id']}:{r['index']}" for r in row['retrieved']])
    ref_set = set([f"{r['document_id']}:{r['index']}" for r in row['reference']])

    return set(ref_set) <= set(ret_set)

def set_abstain(row):
    return (row['generated_answer'] == 'I DO NOT KNOW') or (row['generated_answer'] == 'NOT ENOUGH INFO')

def set_task_success(row):
    return row['correct_answer']

def set_generator_success(row):
    if row['retriever_success'] == True:
        return row['task_success']
    else:
        return row['abstain']

In [7]:
results_all = {}
success_rates = {}

simulate = Simulate()

for g in os.listdir(f"{DS_FEVER}"):
    for experiment in os.listdir(f"{DS_FEVER}/{g}/"):
        id = experiment.split('_')
        results = pd.read_json(f'{DS_FEVER}/{g}/{experiment}/results.json')

        results['dataset'] = 'fever'
        results['generator'] = g
        results['retriever_strategy'] = experiment

        results['correct_query'] = True
        results['retriever_success'] = results.apply(get_retrieval_success, axis=1)
        results['abstain'] = results.apply(set_abstain, axis=1)
        results['task_success'] = results.apply(set_task_success, axis=1)
        results['generator_success'] = results.apply(set_generator_success, axis=1)

        results_all[f"fever_{g}_{experiment}"] = results
        success_rates[f"fever_{g}_{experiment}"] = simulate.compute_uncertainty(results)


for g in os.listdir(f"{DS_HOTPOTQA}"):
    for experiment in os.listdir(f"{DS_HOTPOTQA}/{g}/"):
        id = experiment.split('_')
        results = pd.read_json(f'{DS_HOTPOTQA}/{g}/{experiment}/results.json')

        results['dataset'] = 'hotpotqa'
        results['generator'] = g
        results['retriever_strategy'] = experiment

        results['correct_query'] = True
        results['retriever_success'] = results.apply(get_retrieval_success, axis=1)
        results['abstain'] = results.apply(set_abstain, axis=1)
        results['task_success'] = results.apply(set_task_success, axis=1)
        results['generator_success'] = results.apply(set_generator_success, axis=1)
                            
        results_all[f"hotpotqa_{g}_{experiment}"] = results
        success_rates[f"hotpotqa_{g}_{experiment}"] = simulate.compute_uncertainty(results)

for g in os.listdir(f"{DS_NQ}"):
    for experiment in os.listdir(f"{DS_NQ}/{g}/"):
        try:
            id = experiment.split('_')
            results = pd.read_json(f'{DS_NQ}/{g}/{experiment}/results.json')

            results['dataset'] = 'nq'
            results['generator'] = g
            results['retriever_strategy'] = experiment

            results['correct_query'] = True
            results['retriever_success'] = results.apply(get_retrieval_success, axis=1)
            results['abstain'] = results.apply(set_abstain, axis=1)
            results['task_success'] = results.apply(set_task_success, axis=1)
            results['generator_success'] = results.apply(set_generator_success, axis=1)
                                
            results_all[f"nq_{g}_{experiment}"] = results
            success_rates[f"nq_{g}_{experiment}"] = simulate.compute_uncertainty(results)
        except:
            continue

out = pd.concat((df for df in results_all.values()), ignore_index=True)


In [6]:
t = PrettyTable(field_names=['Dataset', 'Generator', 'Retriever Strategy', 'P(R=1)', 'P(G=1)', 'P(G=1|R=1)', 'P(G=1|R=0)'])

for experiment, success_rate in success_rates.items():
    id = experiment.split('_')

    df = out.loc[(out['dataset'] == id[0]) & (out['generator'] == id[1]) & (out['retriever_strategy'] == id[2])]

    t.add_row([
        id[0],
        id[1],
        id[2],
        f"{df['retriever_success'].mean():.2f}",
        f"{df['generator_success'].mean():.2f}",
        f"{df.loc[df['retriever_success'] == True]['generator_success'].mean():.2f}",
        f"{df.loc[df['retriever_success'] == False]['generator_success'].mean():.2f}"
    ]) 
 
t

Dataset,Generator,Retriever Strategy,P(R=1),P(G=1),P(G=1|R=1),P(G=1|R=0)
fever,gemma3,dense,0.60,0.78,0.94,0.54
fever,gemma3,sparse,0.46,0.74,0.95,0.57
fever,gemma3,hybrid,0.66,0.86,0.94,0.71
fever,apertus,dense,0.60,0.75,0.85,0.61
fever,apertus,sparse,0.46,0.76,0.87,0.67
fever,apertus,hybrid,0.66,0.80,0.85,0.70
fever,qwen,dense,0.60,0.75,0.93,0.48
fever,qwen,sparse,0.46,0.71,0.93,0.52
fever,qwen,hybrid,0.66,0.84,0.93,0.66
hotpotqa,gemma3,dense,0.15,0.25,0.56,0.20


In [8]:
t = PrettyTable(field_names=['Dataset', 'Generator', 'Retriever Strategy', 'P(R=1)', 'P(A=1)', 'P(T=1)', 'P(G=1)'])

for experiment, success_rate in success_rates.items():
    id = experiment.split('_')

    df = out.loc[(out['dataset'] == id[0]) & (out['generator'] == id[1]) & (out['retriever_strategy'] == id[2])]

    t.add_row([
        id[0],
        id[1],
        id[2],
        f"{df['retriever_success'].mean():.2f}",
        f"{df['abstain'].mean():.2f}",
        f"{df['task_success'].mean():.2f}",
        f"{df['generator_success'].mean():.2f}",
        # f"{df.loc[df['retriever_success'] == True]['generator_success'].mean():.2f}",
        # f"{df.loc[df['retriever_success'] == False]['generator_success'].mean():.2f}"
    ]) 
 
t

Dataset,Generator,Retriever Strategy,P(R=1),P(A=1),P(T=1),P(G=1)
fever,gemma3,dense,0.60,0.18,0.78,0.74
fever,gemma3,sparse,0.46,0.22,0.74,0.64
fever,gemma3,hybrid,0.66,0.10,0.86,0.70
fever,apertus,dense,0.60,0.13,0.75,0.61
fever,apertus,sparse,0.46,0.12,0.76,0.50
fever,apertus,hybrid,0.66,0.09,0.80,0.61
fever,qwen,empty,0.00,0.68,0.28,0.68
fever,qwen,oracle,1.00,0.04,0.92,0.92
fever,qwen,dense,0.60,0.20,0.75,0.74
fever,qwen,sparse,0.46,0.25,0.71,0.66
